# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Schema URL:** [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the URL to the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as a single object
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

First, list all record sets defined in this dataset package, referencing by their `@id`. Then explore fields and columns with their `@id` values.

In [ ]:
# List record sets
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    # recordSet may be a list or a single dict
    if isinstance(metadata.recordSet, list):
        record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in metadata.recordSet]
    elif isinstance(metadata.recordSet, dict):
        record_sets = [metadata.recordSet['@id']]
else:
    # Record sets not found in metadata. Try to use 'dataset.records' to iterate available record sets.
    try:
        record_sets = dataset.record_sets()
        record_sets = [rs['@id'] if isinstance(rs, dict) else rs for rs in record_sets]
    except Exception as e:
        print('No record sets found.')

print('Available record set @id values:')
for rs_id in record_sets:
    print(f' - {rs_id}')

# Explore each record set for fields and columns by @id
fields_by_recordset = {}
for rs_id in record_sets:
    try:
        rs = dataset.get_record_set(rs_id)
        fields = rs['field'] if 'field' in rs and rs['field'] else []
        fields_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else f for f in fields]
        columns_ids = []
        if 'column' in rs and rs['column']:
            columns = rs['column']
            columns_ids = [c['@id'] if isinstance(c, dict) and '@id' in c else c for c in columns]
        fields_by_recordset[rs_id] = {
            'fields': fields_ids,
            'columns': columns_ids
        }
        print(f"\nRecord Set @id: {rs_id}")
        print(f"  Field @id values: {fields_ids}")
        print(f"  Column @id values: {columns_ids}")
    except Exception as e:
        print(f'Could not retrieve fields/columns for record set {rs_id}: {e}')

# If no record sets are available, print a notice
if not record_sets:
    print('No record sets discovered in metadata.')

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. Use the record set and field `@id`s discovered above.

If the dataset has multiple record sets, load each; otherwise, load any available one.

In [ ]:
# Load records from each record set into a DataFrame
dataframes = {}
loaded_recordsets = []

for record_set_id in record_sets:
    try:
        records_gen = dataset.records(record_set=record_set_id)
        records = list(records_gen)
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            loaded_recordsets.append(record_set_id)
            print(f"\nLoaded {len(df)} records for record set @id: {record_set_id}")
            print("Columns:", df.columns.tolist())
            print(df.head())
        else:
            print(f'No records found for record set @id: {record_set_id}')
    except Exception as e:
        print(f'Could not load records for record set {record_set_id}: {e}')

# Select a record set for downstream EDA
if loaded_recordsets:
    main_record_set_id = loaded_recordsets[0]
    main_df = dataframes[main_record_set_id]
    print(f"\nUsing record set @id: {main_record_set_id} for further analysis.")
else:
    main_record_set_id = None
    main_df = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. For all data manipulations, reference columns/fields using their `@id` values as keys.

In [ ]:
if main_df is not None:
    # Attempt to select a numeric column by @id
    numeric_field_id = None
    # Search for columns that look numeric in the dataframe
    numeric_candidates = []
    for col in main_df.columns:
        if main_df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_candidates.append(col)
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        # Try common field names
        candidates = ['gad7_score', 'phq9_score', 'psq_score', 'age']
        for candidate in candidates:
            if candidate in main_df.columns:
                numeric_field_id = candidate
                break

    if numeric_field_id is not None:
        print(f"\nSelected numeric field for analysis: {numeric_field_id}")
        threshold = 10
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try grouping by a categorical, demographic field
        group_field_id = None
        demographic_candidate_ids = ['gender', 'village', 'level_of_education', 'marital_status']
        for col in main_df.columns:
            if col in demographic_candidate_ids:
                group_field_id = col
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No main dataframe available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, for example, we plot the distribution of the selected numeric field and, if a group field is available, compare group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field_id is not None:
    plt.figure(figsize=(10,6))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Plot group-wise means if possible
    if group_field_id is not None and group_field_id in main_df.columns:
        plt.figure(figsize=(10,6))
        group_means = main_df.groupby(group_field_id)[numeric_field_id].mean().dropna()
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No numeric or group field to visualize.")

## 6. Conclusion
This notebook demonstrates how to use the `mlcroissant` library to load and explore a survey dataset defined by a Croissant schema. We reviewed the available record sets, fields, and columns via their `@id` values, loaded the data, performed basic EDA including filtering and normalization, and visualized distributions and group-wise means.

Further exploration could include more detailed demographic analysis, inspecting missing data patterns, and feature engineering for modeling tasks.